In [ ]:
import torch
import matplotlib.pyplot as plt
from matplotlib.ticker import ScalarFormatter, NullFormatter
import time

#DEVICE = "cuda"
DEVICE = "mps"

def do_test(iterations, func, data):
    torch.mps.synchronize()
    start = time.time()
    for _ in range(iterations):
        func(data)
    torch.mps.synchronize()
    end = time.time()
    return end - start


def get_f(x, repeat):
    def f(x):
        for _ in range(repeat):
            x = x * 2 + 7
        return x
    return f

X = torch.randn((int(2**26),), device=DEVICE, dtype=torch.float32, requires_grad=False)

#f = get_f(X, 10)
#f = torch.jit.trace(f, X)
#state("before")
#with torch.jit.fuser("fuser2"):

#torch.cuda.synchronize()
#print(torch.jit.last_executed_optimized_graph())

#print(f"repeat={10} {do_test(10, f, X):.4}s")
    

repeats = []
times = []

mem_bend = []
flops_stats = []

def get_log_splits(start, end, n):
    return [int(start * (end/start)**(i/n)) for i in range(n+1)]

xes = get_log_splits(1, 1024, 20)

for n in xes:
    print(f"\n-----------------------\nrepeat={n}:")
    f = get_f(X, n)
    #f = torch.jit.trace(f,X)
    torch._dynamo.reset()
    f = torch.compile(f)
    for _ in range(5): f(X)
    #torch.mps.synchronize()
    #print(torch.jit.last_executed_optimized_graph())
   
    M = 10

    data_size = X.numel() * X.element_size()
    t_iter = do_test(M, f, X)/M # time per iteration
    iter_per_sec = 1 / t_iter  # iterations per second
    memory_per_second = iter_per_sec * data_size * 2
    flops = n * X.numel() * iter_per_sec

    print(f"iter={M} t_per_iter={t_iter:.4}s {iter_per_sec:.2} iter/s {memory_per_second:.2} bytes/s")

    print(f"FLOPS: { flops/1e12:.2f} TF/s")
    print(f"Mem B/W: {memory_per_second/1e9:.2f} GB/s")
    
    repeats.append(n)
    times.append(t_iter*1000)
    mem_bend.append(memory_per_second/1e9)
    flops_stats.append(flops/1e12)

plt.plot(repeats, times, label="Time / iter")
plt.xlabel("fused(ops)")
plt.ylabel("Time (ms)") 
plt.legend()
plt.xscale("log", base=2)
ax = plt.gca()
ax.set_xticks(repeats)
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_formatter(NullFormatter())
ax.ticklabel_format(axis="y", style="plain")
plt.show()

plt.plot(repeats, mem_bend, label="Mem B/W")
plt.xlabel("fused(ops)")
plt.ylabel("Mem B/W (GB/s)") 
plt.legend()
plt.xscale("log", base=2)
ax = plt.gca()
ax.set_xticks(repeats)
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_formatter(NullFormatter())
ax.ticklabel_format(axis="y", style="plain")
plt.show()

plt.plot(repeats, flops_stats, label="FLOPS")
plt.xlabel("fused(ops)")
plt.ylabel("FLOPS (TF/s)") 
plt.legend()
plt.xscale("log", base=2)
ax = plt.gca()
ax.set_xticks(repeats)
ax.xaxis.set_major_formatter(ScalarFormatter())
ax.xaxis.set_minor_formatter(NullFormatter())
ax.ticklabel_format(axis="y", style="plain")
plt.show()



#print(torch.compile(get_f(X, 10)))







-----------------------
repeat=1:


RuntimeError: Cannot execute deviceSynchronize() without MPS backend.

So from what I see, I have on my RTX 3060 peaks:
  - Mem B at around 320 GB/s
  - Compute at ~6 TFLOPS